# Happy-LLM 第五章 · Kaggle GPU 学习版

从随机权重开始：数据 → BPE → 手写 LLaMA 风格 Decoder → 预训练 → SFT → 评估 → 生成。

先开启 GPU 和 Internet，再逐格运行。约 26.75M 参数，默认只使用一张 GPU。这个训练规模用于学习流程，不保证生成质量。完整步骤在解压出来的 README.md。

模型已经按结构拆分：`model/config.py`、`norm.py`、`rope.py`、`attention.py`、`ffn.py`、`decoder.py`、`transformer.py`、`generation.py`。
从 `transformer.py` 看整体，再进入 `decoder.py` 看单层，再看 attention 和 FFN。训练命令不变。

In [ ]:
# 1. 创建项目：源码已内嵌，无需额外上传代码 Dataset
import base64, io, zipfile, os
from pathlib import Path
ROOT = Path('/kaggle/working/happy_llm_kaggle')
ROOT.mkdir(parents=True, exist_ok=True)
(ROOT / 'model.py').unlink(missing_ok=True)  # 清理旧版单文件模型
PAYLOAD = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    archive.extractall(ROOT)
os.chdir(ROOT)
print('项目目录：', ROOT)
print([p.name for p in ROOT.iterdir()])


In [ ]:
# 2. 保留 Kaggle 自带 CUDA PyTorch，只安装外围依赖
import sys, subprocess
requirements = [line for line in Path('requirements.txt').read_text().splitlines() if line and not line.startswith('torch')]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *requirements], check=True)
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=ROOT, check=True)


In [ ]:
# 3. 检查 GPU。失败时不要继续跑训练
import torch
assert torch.cuda.is_available(), '没有检测到 GPU，请在 Kaggle 设置中开启 GPU 加速器后重启会话。'
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('显存 GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)
print('本项目只使用 cuda:0')


## 数据与分词器
默认流式抽样 20,000 条预训练、5,000 条 SFT；训练/验证/测试划分为 90%/5%/5%。这里只训练普通文本对话，工具调用样本会被过滤。不要在训练中途重新运行分词器训练。

In [ ]:
# 4. 下载并划分真实数据；只读取前 N 条有效唯一记录
run('prepare.py', '--pretrain-n', '20000', '--sft-n', '5000')
print(Path('data/manifest.json').read_text())


In [ ]:
# 5. 从预训练训练集训练 BPE
run('train_tokenizer.py')
run('check.py')
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('tokenizer')
print(tok.encode('你好，世界！', add_special_tokens=False))
print(tok.apply_chat_template([{'role':'user','content':'你好'}], tokenize=False, add_generation_prompt=True))


## 从零预训练
输入是前面的 token，标签是后一个 token；padding 不计算损失。默认 500 个优化器更新步，每步有效 batch=4×4。如显存不足，改为 batch=2、accum=8；参数要在运行前确定。

In [ ]:
# 6. 预训练，随机初始化，无现成模型权重
run('train.py', '--stage', 'pretrain', '--steps', '500', '--batch-size', '4', '--accum', '4')


In [ ]:
# 7. 预训练留出集评估及续写
run('evaluate.py', '--checkpoint', 'outputs/pretrain/best.pt', '--stage', 'pretrain', '--out', 'outputs/pretrain_test.json')
run('generate.py', '--checkpoint', 'outputs/pretrain/best.pt', '--prompt', '人工智能是', '--out', 'outputs/base_generation.json')


## SFT：学习如何回答
从预训练的 best.pt 初始化，进行全参数训练；仅回答正文和结束 token 计算损失。该规模不保证产生可用聊天能力。

In [ ]:
# 8. 在同一个 SFT 测试集上记录微调前基线
run('evaluate.py', '--checkpoint', 'outputs/pretrain/best.pt', '--stage', 'sft', '--out', 'outputs/base_on_sft_test.json')
run('train.py', '--stage', 'sft', '--init', 'outputs/pretrain/best.pt', '--steps', '200', '--lr', '1e-4', '--batch-size', '4', '--accum', '4')


In [ ]:
# 9. 同一测试集上评估微调后模型，并生成回答
run('evaluate.py', '--checkpoint', 'outputs/sft/best.pt', '--stage', 'sft', '--out', 'outputs/sft_test.json')
run('generate.py', '--checkpoint', 'outputs/sft/best.pt', '--chat', '--prompt', '什么是人工智能？', '--out', 'outputs/sft_generation.json')
import json
for filename in ['outputs/base_on_sft_test.json', 'outputs/sft_test.json']:
    print(filename, json.loads(Path(filename).read_text()))


In [ ]:
# 10. 绘制训练曲线
run('plot.py')
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'outputs/loss.png')))


In [ ]:
# 11. 导出所有结果。请下载，并保存 Kaggle Notebook 版本及输出
import shutil
result = shutil.make_archive('/kaggle/working/happy_llm_result', 'zip', root_dir=ROOT)
print('结果包：', result)
from IPython.display import FileLink
display(FileLink('../happy_llm_result.zip'))


## 断点恢复（仅需要时运行）
预训练恢复：`run("train.py", "--stage", "pretrain", "--resume", "outputs/pretrain/last.pt", "--steps", "500", "--batch-size", "4", "--accum", "4")`。

SFT 恢复：`run("train.py", "--stage", "sft", "--resume", "outputs/sft/last.pt", "--steps", "200", "--lr", "1e-4", "--batch-size", "4", "--accum", "4")`。保持原数据、tokenizer 和配置。会话结束后先把结果包重新加入输入并解压。只加载自己生成的可信 checkpoint。